In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.checkpoint.sqlite import SqliteSaver
from typing import TypedDict, Annotated
from dotenv import load_dotenv
import sqlite3

In [2]:
load_dotenv()

True

In [3]:
model=ChatGroq(model="openai/gpt-oss-120b")

In [4]:
from langgraph.graph.message import add_messages
class chatstate(TypedDict):
    messages:Annotated[list[BaseMessage], add_messages ] #adda-msg is reducer function

In [5]:
def chat_node(state:chatstate)->chatstate:

    #take user querry from state
    message=state['messages']

    #send to llm
    response=model.invoke(message)

    #response store stste
    return {
        'messages': [response]
    }

#make database first
conn=sqlite3.connect(database='database.db', check_same_thread=False)

graph=StateGraph(chatstate)

#node
graph.add_node('chat_node', chat_node)

#edges
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

checpointer= SqliteSaver(conn=conn)
chatbot=graph.compile(checkpointer=checpointer)


In [6]:
while True:
    user=input('ask anything :')
    print('user:', user)
    if user.strip().lower() in ['quit', 'stop' ]:
        break

    #config : 1
    config1= {'configurable':{'thread_id': '1'}}
    
    #config : 2
    config2= {'configurable':{'thread_id': '2'}}

    #yaha phar jo coversation start karna chahe start karsakte hun bs config name do jo define hai
    answer=chatbot.invoke({'messages':[HumanMessage(content=user)] }, config=config2)

    print('AI :', answer['messages'][-1].content)

user: hi what is my name 
AI : Your name is Awais. 😊 How can I help you today?
user: stop
